# Notebook 02
**Pergunta central do Data Storytelling:**
> "O que faz um livro ser verdadeiramente bom? Em um mar de milhões de títulos, a reputação de uma obra muitas vezes parece um mistério. Seria o sucesso definido pela 'sabedoria das multidões' (quando milhões de pessoas avaliam algo positivamente)? Ou seria a genialidade consistente de um autor de elite?"*
---

Para responder a essa pergunta, este notebook conduz duas linhas de análise complementares:

1. Análise do Paradoxo da Popularidade: Cruzar a nota média com o volume de avaliações para investigar se os livros mais famosos (blockbusters) são realmente os mais bem avaliados, ou se as notas perfeitas se escondem na "cauda longa" de nicho.

2. Análise da Consistência de Autores: Identificar a "elite" da plataforma, avaliando quais escritores conseguem manter o mais alto padrão de qualidade em múltiplas obras publicadas, servindo como uma "aposta segura" para recomendações.

---
### Observações importantes:
* Uso de Escala Logarítmica: Na análise de popularidade, utilizamos o eixo Y em escala logarítmica. Isso foi feito para evitar que a visualização de 99% dos livros fosse esmagada pelos mega best-sellers (outliers com milhões de avaliações), permitindo uma comparação justa.
* Filtro de Relevância de Autores: Para o ranking de melhores autores, adotamos um ponto de corte técnico exigindo um mínimo de 5 livros publicados. Essa métrica foi aplicada para eliminar one-hit wonders (autores de um único grande sucesso) e focar na consistência estatística a longo prazo.

In [4]:
# Preparação do Ambiente e Carga de Dados
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import numpy as np

# Carregando nossa 'biblioteca' de dados
try:
    df = pd.read_parquet('datasets/books.parquet')
    print(f"Sucesso! Base carregada com {df.shape[0]} títulos prontos para análise.")
except Exception as e:
    print(f"Erro ao carregar os dados: {e}")


Sucesso! Base carregada com 84054 títulos prontos para análise.


## 1. O Paradoxo da Popularidade: Qualidade vs. Quantidade

Ao analisarmos a relação entre a nota média e o volume de avaliações, nos deparamos com um fenômeno interessante: o Paradoxo da Popularidade.

Livros com notas perfeitas tendem a ser obras de nicho, enquanto os grandes blockbusters convergem para uma média mais 'pé no chão'. Para enxergar essa disparidade, utilizamos uma escala logarítmica no eixo de avaliações.

In [5]:
# Filtramos livros com pelo menos 1 avaliação para evitar ruído
col_rating = 'rating' if 'rating' in df.columns else 'average_rating'
col_totalratings = 'totalratings' if 'totalratings' in df.columns else 'ratings_count'

df_pop = df[df[col_totalratings] > 0].copy()

fig1 = px.scatter(
    df_pop, 
    x=col_rating, 
    y=col_totalratings, 
    log_y=True, 
    title='Popularidade (Qtd Avaliações) vs. Qualidade (Nota Média)',
    labels={col_rating: 'Nota Média', col_totalratings: 'Total de Avaliações (Escala Log)'},
    color=col_rating,
    color_continuous_scale='Reds',
    template='plotly_white'
)
fig1.show()


## 2. Em Busca da Consistência: Os Autores de Elite

A popularidade pode ser passageira, mas a consistência é o que define um mestre. 

Decidimos investigar: quais autores conseguem manter um alto padrão de qualidade em múltiplas obras? Para isso, filtramos apenas aqueles com pelo menos 5 livros publicados.

In [16]:
# Identificando os autores consistentes
col_author = 'author' if 'author' in df.columns else 'authors'
col_rating = 'rating' if 'rating' in df.columns else 'average_rating'
col_totalratings = 'totalratings' if 'totalratings' in df.columns else 'ratings_count'

# Caso o autor não tenha sido separado no notebook_00, vamos separar aqui
df_authors = df.assign(author_temp=df[col_author].str.split(',')).explode('author_temp')
df_authors['author_temp'] = df_authors['author_temp'].str.strip()

contagem_autores = df_authors['author_temp'].value_counts()
autores_elite_lista = contagem_autores[contagem_autores >= 5].index
df_elite = df_authors[df_authors['author_temp'].isin(autores_elite_lista)]

author_stats = df_elite.groupby('author_temp').agg(
    Media_Nota=(col_rating, 'mean'),
    Qtd_Livros=('title', 'count'),
    Total_Avaliacoes=(col_totalratings, 'sum')
).reset_index()

# Filtro de popularidade: Mínimo 500.000 avaliações totais para autores de elite
author_stats = author_stats[author_stats['Total_Avaliacoes'] >= 500000]
author_stats = author_stats.sort_values(by='Media_Nota', ascending=False).head(10)

fig2 = px.bar(
    author_stats, 
    x='Media_Nota', 
    y='author_temp', 
    orientation='h',
    title='Top 10 Autores com Melhor Média de Notas (Mín. 5 livros)',
    labels={'Media_Nota': 'Nota Média Geral', 'author_temp': 'Autor'},
    hover_data=['Qtd_Livros']
)

fig2.update_traces(marker_color='#6F1D1B')
fig2.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_range=[3.4, 5])
fig2.show()

## Conclusão: Padrões de Qualidade e Engajamento

Neste notebook, exploramos variáveis fundamentais para a construção de um sistema de recomendação robusto para o aplicativo Booklog. As análises interativas nos permitiram extrair duas conclusões principais:

### 1. O Paradoxo da Popularidade vs. Qualidade: 
Observamos que os livros com as maiores notas absolutas (próximas a 5.0) geralmente pertencem à "cauda longa", ou seja, são obras de nicho com pouquíssimas avaliações. Em contrapartida, os blockbusters (com milhões de avaliações) tendem a ter suas notas estabilizadas em uma média mais realista (entre 3.8 e 4.5). Impacto no App: Nosso futuro algoritmo não poderá recomendar livros baseando-se apenas na nota máxima bruta, pois acabaria indicando obras desconhecidas. Será necessário criar um "peso" que equilibre a nota com a quantidade de leitores.

---

### 2. A Consistência dos Autores de Elite: 
Identificamos que existe um grupo seleto de autores que, mesmo com um volume alto de publicações (5+ livros), conseguem manter médias de avaliação altíssimas. Impacto no App: Estes autores representam o que chamamos de "Apostas Seguras" (Safe Bets). Eles serão excelentes candidatos para solucionar o problema de Cold Start (quando um usuário novo entra no app e ainda não sabemos o que ele gosta, recomendamos esses mestres da literatura).
